### ЗАДАЧА: Резервирование товаров по складам

Команда закупок получает пакет заявок на резерв товаров по складам.
Нужно собрать систему, которая:
- принимает корректные заявки,
- отбрасывает неправильные или невыполнимые запросы,
- обновляет остатки на складе после успешного резерва,
- собирает отдельный журнал ошибок,
- позволяет быстро понять, кто зарезервировал больше всего товара и на каком складе осталось меньше всего позиций.

В части строк указаны неизвестные склады или товары,
в части количество заполнено неверно,
а некоторые заявки пытаются зарезервировать больше доступного остатка или дублируют уже обработанный `request_id`.


In [ ]:
from dataclasses import dataclass
from typing import Optional

stocks = {
    'MSK-1': {'keyboard': 10, 'mouse': 20, 'monitor': 4},
    'SPB-2': {'keyboard': 6, 'dock': 5, 'monitor': 2},
    'KZN-3': {'mouse': 7, 'dock': 3, 'laptop': 2},
}

# rows: request_id|client|warehouse_id|sku|quantity
rows = [
    'RQ-100|Acme|MSK-1|keyboard|3',
    'RQ-101|Beta|SPB-2|dock|2',
    'RQ-102|Acme|MSK-1|monitor|5',
    'RQ-103|Delta|X-999|mouse|1',
    'RQ-104|Gamma|KZN-3|laptop|0',
    'RQ-105|Beta|SPB-2|chair|1',
    'RQ-101|Beta|MSK-1|mouse|4',
    'RQ-106|Acme|MSK-1|mouse|7',
    'RQ-107|Kira|KZN-3|laptop|1',
]

class ReservationError(Exception):
    pass

class RowFormatError(ReservationError):
    pass

class WarehouseNotFoundError(ReservationError):
    pass

class ProductNotFoundError(ReservationError):
    pass

class QuantityError(ReservationError):
    pass

class StockLimitError(ReservationError):
    pass

class DuplicateRequestError(ReservationError):
    pass

@dataclass(order=True)
class ReservationRequest:
    request_id: str
    client: str
    warehouse_id: str
    sku: str
    quantity: int

class Warehouse:
    def __init__(self, warehouse_id, products):
        # TODO: сохранить warehouse_id
        self.warehouse_id = warehouse_id
        # TODO: создать отдельную копию словаря products
        self.products = dict(products)
        # TODO: создать список reservations
        self.reservations: list[ReservationRequest] = []
        
    def has_sku(self, sku):
        # TODO: вернуть True/False, есть ли такой sku в self.products
        return sku in self.products

    def available(self, sku: str) -> int:
        # TODO: вернуть текущий остаток по sku
        return self.products.get(sku, 0)

    def reserve(self, request):
        # TODO: если request.sku отсутствует -> raise ProductNotFoundError(...)
        if request.sku not in self.products:
            raise ProductNotFoundError(f"Товар '{request.sku}' не найден на складе '{self.warehouse_id}'")
        
        # TODO: если request.quantity > available(...) -> raise StockLimitError(...)
        available_qty = self.available(request.sku)
        if request.quantity > available_qty:
            raise StockLimitError(f"Недостаточно товара '{request.sku}': доступно {available_qty}, запрошено {request.quantity}")
        
        # TODO: уменьшить остаток на складе
        self.products[request.sku] -= request.quantity
        # TODO: добавить request в self.reservations
        self.reservations.append(request)
        
    def total_left(self):
        # TODO: вернуть сумму всех остатков на складе
        return sum(self.products.values())

    def reserved_total(self):
        # TODO: вернуть сумму quantity по всем self.reservations
        return sum(r.quantity for r in self.reservations)

class ReservationService:
    def __init__(self, stocks):
        # TODO: создать warehouses вида warehouse_id -> Warehouse(...)
        self.warehouses = {warehouse_id: Warehouse(warehouse_id, products) for warehouse_id, products in stocks.items()}
        # TODO: создать списки accepted и errors
        self.accepted = []
        self.errors = []
        # TODO: создать множество processed_ids
        self.processed_ids = set()
        
    def parse_request(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: request_id, client, warehouse_id, sku, quantity_raw
        parts = row.split('|')
        if len(parts) != 5:
            raise RowFormatError("Неверный формат записи")
        request_id, client, warehouse_id, sku, quantity_raw = parts
        # TODO: quantity_raw преобразовать в int
        try:
            quantity = int(quantity_raw)
        except ValueError:
            raise QuantityError(f"Количество должно быть целым числом: '{quantity_raw}'")
        # TODO: если quantity <= 0 -> QuantityError
        if quantity <= 0:
            raise QuantityError(f"Количество должно быть положительным: {quantity}")
        # TODO: если warehouse_id не существует -> WarehouseNotFoundError
        if warehouse_id not in self.warehouses:
            raise WarehouseNotFoundError(f"Склад '{warehouse_id}' не найден")
        # TODO: вернуть объект ReservationRequest(...)
        return ReservationRequest(request_id, client, warehouse_id, sku, quantity)
        
    def submit(self, row):
        # TODO: внутри try вызвать parse_request(row)
        try:
            request = self.parse_request(row)
            # TODO: если request.request_id уже в processed_ids -> DuplicateRequestError
            if request.request_id in self.processed_ids:
                raise DuplicateRequestError(f"Заявка с ID '{request.request_id}' уже обработана")
            # TODO: затем warehouses[request.warehouse_id].reserve(request)
            warehouse = self.warehouses[request.warehouse_id]
            warehouse.reserve(request)
            # TODO: после успеха добавить request_id в processed_ids
            self.processed_ids.add(request.request_id)
            # TODO: добавить request в self.accepted
            self.accepted.append(request)
        # TODO: ReservationError сохранить в self.errors как (row, error_type, message)
        except ReservationError as e:
            self.errors.append((row, type(e).__name__, str(e)))

    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for row in rows:
            self.submit(row)

    def client_totals(self):
        # TODO: собрать dict вида client -> total_reserved_quantity
        totals = {}
        for request in self.accepted:
            totals[request.client] = totals.get(request.client, 0) + request.quantity
        return totals

    def top_client(self):
        # TODO: использовать client_totals()
        # TODO: вернуть tuple(client, total_quantity) с максимумом
        client_totals = self.client_totals()
        if not client_totals:
            return None
        top_client = max(client_totals.items(), key=lambda x: x[1])
        return top_client

    def lowest_stock_warehouse(self):
        # TODO: найти склад с минимумом total_left()
        # TODO: вернуть tuple(warehouse_id, total_left)
        if not self.warehouses:
            return None
        lowest = min(self.warehouses.items(), key=lambda x: x[1].total_left())
        return (lowest[0], lowest[1].total_left())

    def warehouse_snapshot(self):
        # TODO: собрать dict вида warehouse_id -> копия текущих остатков products
        snapshot = {}
        for warehouse_id, warehouse in self.warehouses.items():
            snapshot[warehouse_id] = dict(warehouse.products)
        return snapshot

    def find_request(self, request_id) -> Optional[ReservationRequest]:
        # TODO: вернуть Optional[ReservationRequest]
        # TODO: пройтись по self.accepted и найти нужную заявку
        # TODO: если не найдено -> вернуть None
        for request in self.accepted:
            if request.request_id == request_id:
                return request
        return None

# Использование сервиса
service = ReservationService(stocks)
# TODO: загрузить rows через service.load(rows)
service.load(rows)

# TODO: вывести принятые заявки
print("Принятые заявки:")
for req in service.accepted:
    print(f"  {req}")

# TODO: вывести ошибки
print("\nОшибки:")
for error in service.errors:
    print(f"  Строка: {error[0]} | Ошибка: {error[1]} | Сообщение: {error[2]}")

# TODO: вывести warehouse_snapshot()
print(f"\nСостояние складов: {service.warehouse_snapshot()}")

# TODO: вывести client_totals()
print(f"Суммарные заказы по клиентам: {service.client_totals()}")

# TODO: вывести top_client()
print(f"Клиент с наибольшим количеством заказов: {service.top_client()}")

# TODO: вывести lowest_stock_warehouse()
print(f"Склад с наименьшим остатком: {service.lowest_stock_warehouse()}")

# TODO: вывести find_request('RQ-107')
print(f"Поиск заявки RQ-107: {service.find_request('RQ-107')}")